In [1]:
import pandas as pd
import csv
import os
import requests
from io import StringIO
import re
import json

In [2]:
url = "https://raw.githubusercontent.com/anilbhaila/llm-zoomcamp-finalproject/refs/heads/main/data/Ecommerce_FAQ_Chatbot_dataset.json"


In [3]:
def load_data(*args, **kwargs):
    """
    Extract data from URL. 
    
    """
    
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        
        json_data = response.json()

        faqs = json_data.get("questions")
        # Create a DataFrame
        df = pd.DataFrame(list(faqs))

        return df
    except Exception as e:
        print(f"An error occurred while reading the CSV file: {e}")
        return None

In [5]:
data = load_data()
data.head()

,question,answer
0,How can I create an account?,"To create an account, click on the 'Sign Up' b..."
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and..."
2,How can I track my order?,You can track your order by logging into your ...
3,What is your return policy?,Our return policy allows you to return product...
4,Can I cancel my order?,You can cancel your order if it has not been s...


In [11]:
import re

data['sanitized_question'] = data['question'].apply(lambda x: re.sub(r'\W', '_', x[:30]).lower())
data['document_id'] = "doc_" + data.index.astype(str) + "_" + data['sanitized_question']

data

,question,answer,sanitized_question,document_id
0,How can I create an account?,"To create an account, click on the 'Sign Up' b...",how_can_i_create_an_account_,doc_0_how_can_i_create_an_account_
1,What payment methods do you accept?,"We accept major credit cards, debit cards, and...",what_payment_methods_do_you_ac,doc_1_what_payment_methods_do_you_ac
2,How can I track my order?,You can track your order by logging into your ...,how_can_i_track_my_order_,doc_2_how_can_i_track_my_order_
3,What is your return policy?,Our return policy allows you to return product...,what_is_your_return_policy_,doc_3_what_is_your_return_policy_
4,Can I cancel my order?,You can cancel your order if it has not been s...,can_i_cancel_my_order_,doc_4_can_i_cancel_my_order_
...,...,...,...,...
74,Can I order a product if it is listed as 'sold...,If a product is listed as 'sold out' but avail...,can_i_order_a_product_if_it_is,doc_74_can_i_order_a_product_if_it_is
75,Can I return a product if it was purchased wit...,"Yes, you can return a product purchased with a...",can_i_return_a_product_if_it_w,doc_75_can_i_return_a_product_if_it_w
76,Can I request a product if it is not currently...,If a product is not available in your preferre...,can_i_request_a_product_if_it_,doc_76_can_i_request_a_product_if_it_
77,Can I order a product if it is listed as 'comi...,If a product is listed as 'coming soon' but no...,can_i_order_a_product_if_it_is,doc_77_can_i_order_a_product_if_it_is


In [27]:
data_gen_instructions = """
You emulate a ecommerce customer who is asking questions.
Formulate 5 questions this customer might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [13]:
import dotenv
dotenv.load_dotenv("../.env")

True

In [30]:
from openai import OpenAI
openai_client = OpenAI()

In [55]:
documents = data.to_dict(orient='records')

doc = documents[0]
doc

{'question': 'How can I create an account?',
 'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.",
 'sanitized_question': 'how_can_i_create_an_account_',
 'document_id': 'doc_0_how_can_i_create_an_account_'}

In [56]:
import json

user_prompt = json.dumps(doc)
user_prompt

'{"question": "How can I create an account?", "answer": "To create an account, click on the \'Sign Up\' button on the top right corner of our website and follow the instructions to complete the registration process.", "sanitized_question": "how_can_i_create_an_account_", "document_id": "doc_0_how_can_i_create_an_account_"}'

In [19]:
from tqdm.auto import tqdm

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [45]:
messages = [
        {"role": "developer", "content": data_gen_instructions},
        {"role": "user", "content": user_prompt}
    ]

In [46]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [47]:
result = response.output_parsed

print(result)

questions=['How do I sign up for a new account on your website?', 'Where can I find the button to register an account?', 'What’s the fastest way to make an account here?', 'Can you tell me how to create an account from the homepage?', 'How do I complete the registration process and get started?']


In [48]:
import sys
import os

# 1. Direct Python to the root project folder
sys.path.append(os.path.abspath(os.path.join('..')))

from scripts.evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['How do I sign up for an account on your website?', 'Where can I find the button to create a new account?', 'What steps do I need to follow to register on the site?', 'How do I make an account from the top right corner of the page?', 'Is there a quick way to complete the sign-up process?']


In [49]:
usage

ResponseUsage(input_tokens=209, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=80, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=289)

In [50]:
from scripts.evaluation_utils import calc_price
calc_price(usage)

{'input_cost': 0.00015675000000000002,
 'output_cost': 0.00036,
 'total_cost': 0.0005167500000000001}

In [57]:
records = []

for q in result.questions:
    document_id = doc["document_id"]
    records.append({
        "question": q,
        "document": document_id
    })

records

[{'question': 'How do I sign up for an account on your website?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'Where can I find the button to create a new account?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'What steps do I need to follow to register on the site?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'How do I make an account from the top right corner of the page?',
  'document': 'doc_0_how_can_i_create_an_account_'},
 {'question': 'Is there a quick way to complete the sign-up process?',
  'document': 'doc_0_how_can_i_create_an_account_'}]

In [58]:
import pandas as pd

pd.DataFrame(records)

,question,document
0,How do I sign up for an account on your website?,doc_0_how_can_i_create_an_account_
1,Where can I find the button to create a new ac...,doc_0_how_can_i_create_an_account_
2,What steps do I need to follow to register on ...,doc_0_how_can_i_create_an_account_
3,How do I make an account from the top right co...,doc_0_how_can_i_create_an_account_
4,Is there a quick way to complete the sign-up p...,doc_0_how_can_i_create_an_account_


In [59]:
from scripts.evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["document_id"]
        })

    return results, usage

In [60]:
generate_ground_truth(doc)

([{'question': 'How do I sign up for a new account on your website?',
   'document': 'doc_0_how_can_i_create_an_account_'},
  {'question': 'Where can I find the button to register an account?',
   'document': 'doc_0_how_can_i_create_an_account_'},
  {'question': 'What do I need to do after clicking Sign Up to finish creating my account?',
   'document': 'doc_0_how_can_i_create_an_account_'},
  {'question': 'Can I create an account from the top right corner of the site?',
   'document': 'doc_0_how_can_i_create_an_account_'},
  {'question': 'How do I complete the registration process for a new account?',
   'document': 'doc_0_how_can_i_create_an_account_'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=289))

In [ ]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

In [61]:
from concurrent.futures import ThreadPoolExecutor
from scripts.evaluation_utils import map_progress

In [62]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/79 [00:00<?, ?it/s]

In [63]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

395

In [64]:
from scripts.evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.04454625

In [65]:
df_ground_truth = pd.DataFrame(ground_truth)

In [66]:
df_ground_truth

,question,document
0,How do I sign up for an account on your website?,doc_0_how_can_i_create_an_account_
1,Where can I find the Sign Up button to make a ...,doc_0_how_can_i_create_an_account_
2,What steps do I need to follow to register a n...,doc_0_how_can_i_create_an_account_
3,Can you tell me how to create an account on th...,doc_0_how_can_i_create_an_account_
4,How do I complete the registration process aft...,doc_0_how_can_i_create_an_account_
...,...,...
390,"If I bought something during a promo sale, can...",doc_78_can_i_return_a_product_if_it_w
391,Will I get refunded for the discounted price i...,doc_78_can_i_return_a_product_if_it_w
392,Are returns allowed for products purchased on ...,doc_78_can_i_return_a_product_if_it_w
393,"If I return an item I got with a discount, how...",doc_78_can_i_return_a_product_if_it_w


In [67]:
df_ground_truth.to_csv("../data/ground_truth_NEW.csv", index=False)